# 📊 Datathon PEDE - Análise Complementar (Perguntas 6-11)

## Associação Passos Mágicos

Este notebook complementa a análise exploratória respondendo às perguntas 6-11 do desafio.

In [ ]:
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Configurações
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

print('✅ Bibliotecas importadas!')

In [ ]:
# Carregar dados
df = pd.read_excel('../data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx')
print(f"Dataset: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

## 🎯 PERGUNTA 6: Aspectos Psicopedagógicos (IPP)

**Os aspectos psicopedagógicos confirmam ou contradizem a defasagem observada no IAN?**

In [ ]:
print("📊 ANÁLISE DOS ASPECTOS PSICOPEDAGÓGICOS")
print("="*80)

# Verificar se existe coluna IPP ou similar
ipp_cols = [col for col in df.columns if 'IPP' in col.upper() or 'PSICOPEDAG' in col.upper()]
print(f"\nColunas relacionadas a psicopedagogia: {ipp_cols}")

# Como não temos IPP diretamente, vamos usar as recomendações dos avaliadores
rec_cols = [col for col in df.columns if 'Rec' in col]
print(f"Colunas de recomendações: {rec_cols}")

# Análise da relação entre IAN e recomendações
if 'Rec Psicologia' in df.columns:
    print("\n📈 Relação entre IAN e Recomendação de Psicologia:")
    
    # Categorizar IAN
    df['Cat_IAN'] = pd.cut(df['IAN'], bins=[0, 5, 7, 10], labels=['Defasado', 'Moderado', 'Adequado'])
    
    # Análise cruzada
    if df['Rec Psicologia'].notna().sum() > 0:
        cross_tab = pd.crosstab(df['Cat_IAN'], df['Rec Psicologia'], normalize='index') * 100
        print(cross_tab)
    else:
        print("Dados de Rec Psicologia não disponíveis")

# Análise usando número de avaliações
if 'Nº Av' in df.columns:
    print("\n📊 Relação entre IAN e Número de Avaliações:")
    ian_by_av = df.groupby('Nº Av')['IAN'].agg(['mean', 'median', 'count'])
    print(ian_by_av)

In [ ]:
# Visualização
if 'Cat_IAN' in df.columns and 'Nº Av' in df.columns:
    fig = px.box(df, x='Cat_IAN', y='Nº Av',
                 title='Número de Avaliações por Categoria de IAN',
                 labels={'Cat_IAN': 'Categoria IAN', 'Nº Av': 'Número de Avaliações'})
    fig.show()

### 💡 Insights - Pergunta 6 (IPP)

*(Preencher após execução)*

## 🎯 PERGUNTA 7: Ponto de Virada (IPV)

**Quais comportamentos ou combinações de indicadores mais influenciam o IPV (Ponto de Virada)?**

In [ ]:
print("📊 ANÁLISE DO PONTO DE VIRADA (IPV)")
print("="*80)

# Correlações com IPV
indicadores = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV', 'INDE 22']
indicadores_disp = [i for i in indicadores if i in df.columns]

corr_ipv = df[indicadores_disp].corr()['IPV'].sort_values(ascending=False)
print("\n🔗 Correlações com IPV (ordenadas):")
print(corr_ipv)

# Análise de quem atingiu PV
if 'Atingiu PV' in df.columns:
    print("\n📈 Comparação: Atingiu vs Não Atingiu PV")
    
    pv_comparison = df.groupby('Atingiu PV')[indicadores_disp].mean()
    print(pv_comparison)
    
    # Diferença percentual
    if len(pv_comparison) == 2:
        diff = ((pv_comparison.iloc[1] - pv_comparison.iloc[0]) / pv_comparison.iloc[0] * 100)
        print("\n📊 Diferença Percentual (Atingiu vs Não Atingiu):")
        print(diff.sort_values(ascending=False))

In [ ]:
# Visualização das correlações
fig = px.bar(x=corr_ipv.index, y=corr_ipv.values,
             title='Correlação dos Indicadores com IPV',
             labels={'x': 'Indicador', 'y': 'Correlação'},
             color=corr_ipv.values,
             color_continuous_scale='RdYlGn')
fig.show()

In [ ]:
# Análise de combinações (IDA + IEG)
print("\n🔍 Análise de Combinações de Indicadores")
print("="*80)

# Criar categorias combinadas
df['IDA_cat'] = pd.cut(df['IDA'], bins=[0, 5, 7, 10], labels=['Baixo', 'Médio', 'Alto'])
df['IEG_cat'] = pd.cut(df['IEG'], bins=[0, 6, 8, 10], labels=['Baixo', 'Médio', 'Alto'])

# IPV médio por combinação
ipv_by_combination = df.groupby(['IDA_cat', 'IEG_cat'])['IPV'].agg(['mean', 'count'])
print("\nIPV médio por combinação IDA x IEG:")
print(ipv_by_combination)

### 💡 Insights - Pergunta 7 (IPV)

*(Preencher após execução)*

## 🎯 PERGUNTA 8: Multidimensionalidade do INDE

**Quais combinações de indicadores (IAN, IDA, IEG, IAA, IPS, IPP, IPV) explicam melhor o INDE?**

In [ ]:
print("📊 ANÁLISE MULTIDIMENSIONAL DO INDE")
print("="*80)

# Correlações com INDE
if 'INDE 22' in df.columns:
    corr_inde = df[indicadores_disp].corr()['INDE 22'].sort_values(ascending=False)
    print("\n🔗 Correlações com INDE (ordenadas):")
    print(corr_inde)
    
    # Regressão múltipla simples
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score
    
    # Preparar dados
    features = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV']
    features_disp = [f for f in features if f in df.columns]
    
    X = df[features_disp].dropna()
    y = df.loc[X.index, 'INDE 22']
    
    # Treinar modelo
    model = LinearRegression()
    model.fit(X, y)
    
    # Coeficientes
    coef_df = pd.DataFrame({
        'Feature': features_disp,
        'Coeficiente': model.coef_
    }).sort_values('Coeficiente', ascending=False, key=abs)
    
    print("\n📈 Coeficientes da Regressão Linear (INDE):")
    print(coef_df)
    
    # R²
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    print(f"\nR² Score: {r2:.4f}")
    print(f"Variância explicada: {r2*100:.2f}%")

In [ ]:
# Visualização dos coeficientes
fig = px.bar(coef_df, x='Feature', y='Coeficiente',
             title='Importância dos Indicadores na Composição do INDE',
             labels={'Coeficiente': 'Coeficiente de Regressão'},
             color='Coeficiente',
             color_continuous_scale='RdYlGn')
fig.show()

### 💡 Insights - Pergunta 8 (INDE)

*(Preencher após execução)*

## 🎯 PERGUNTA 9: Modelo Preditivo

**É possível criar um modelo preditivo que identifique alunos em risco de defasagem antes que ela se agrave?**

In [ ]:
print("🤖 MODELO PREDITIVO DE RISCO")
print("="*80)

print("\n✅ MODELO JÁ DESENVOLVIDO!")
print("\nDetalhes do Modelo:")
print("  - Algoritmo: XGBoost Classifier")
print("  - Features: 9 (IDA, IEG, IAA, IPS, IPV, Idade, Defas, Gênero, Fase)")
print("  - Target: Risco de Defasagem (Alto/Médio/Baixo)")
print("  - Dataset: 860 alunos (80% treino, 20% teste)")
print("  - Validação: 5-fold Cross-Validation")

print("\n📊 Critérios de Classificação:")
print("  - Alto Risco: IAN < 5 OU (IDA < 5 E IEG < 6)")
print("  - Médio Risco: IAN entre 5-7 OU IDA entre 5-7")
print("  - Baixo Risco: IAN >= 7 E IDA >= 7")

print("\n🎯 Modelo Integrado na Aplicação Streamlit:")
print("  ✅ Predições em tempo real")
print("  ✅ Probabilidades por classe")
print("  ✅ Recomendações personalizadas")
print("  ✅ Visualizações interativas")

# Carregar e mostrar feature importance
import joblib
try:
    model = joblib.load('../models/xgb_model.joblib')
    feature_info = joblib.load('../models/feature_info.joblib')
    
    importance_df = pd.DataFrame({
        'Feature': feature_info['all_features'],
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n📈 Feature Importance (Top 5):")
    print(importance_df.head())
    
except:
    print("\n⚠️ Modelo não encontrado. Execute: python train_model.py")

### 💡 Insights - Pergunta 9 (Modelo Preditivo)

**Resposta: SIM! ✅**

Foi desenvolvido com sucesso um modelo preditivo XGBoost que:
- Identifica alunos em risco de defasagem com base em indicadores atuais
- Classifica o risco em 3 níveis (Alto/Médio/Baixo)
- Fornece probabilidades para cada classe
- Permite intervenção preventiva antes do agravamento
- Está integrado em aplicação web para uso prático

## 🎯 PERGUNTA 10: Efetividade do Programa

**Os indicadores mostram melhora consistente ao longo das fases do programa?**

In [ ]:
print("📊 ANÁLISE DA EFETIVIDADE DO PROGRAMA")
print("="*80)

if 'Fase' in df.columns:
    # Evolução dos indicadores por fase
    indicadores_principais = ['IDA', 'IEG', 'IPV', 'INDE 22']
    indicadores_disp = [i for i in indicadores_principais if i in df.columns]
    
    evolucao = df.groupby('Fase')[indicadores_disp].agg(['mean', 'std', 'count'])
    print("\n📈 Evolução dos Indicadores por Fase:")
    print(evolucao)
    
    # Calcular tendência (regressão linear simples)
    from scipy.stats import linregress
    
    print("\n📊 Análise de Tendência (Coeficiente Angular):")
    fases = sorted(df['Fase'].unique())
    
    for ind in indicadores_disp:
        medias = df.groupby('Fase')[ind].mean()
        slope, intercept, r_value, p_value, std_err = linregress(medias.index, medias.values)
        
        tendencia = "📈 Crescente" if slope > 0.1 else "📉 Decrescente" if slope < -0.1 else "➡️ Estável"
        print(f"  {ind}: {slope:.4f} {tendencia} (R²={r_value**2:.3f}, p={p_value:.4f})")
    
    # Análise de % atingiu PV por fase
    if 'Atingiu PV' in df.columns:
        print("\n🎯 % que Atingiu Ponto de Virada por Fase:")
        pv_por_fase = df.groupby('Fase')['Atingiu PV'].apply(
            lambda x: (x == 'Sim').sum() / len(x) * 100 if len(x) > 0 else 0
        )
        print(pv_por_fase)

In [ ]:
# Visualização da evolução
if 'Fase' in df.columns:
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=indicadores_disp)
    
    for i, ind in enumerate(indicadores_disp):
        row = i // 2 + 1
        col = i % 2 + 1
        
        medias = df.groupby('Fase')[ind].mean()
        
        fig.add_trace(
            go.Scatter(x=medias.index, y=medias.values,
                      mode='lines+markers', name=ind,
                      line=dict(width=3), marker=dict(size=10)),
            row=row, col=col
        )
    
    fig.update_layout(height=600, showlegend=False,
                      title_text="Evolução dos Indicadores por Fase do Programa")
    fig.show()

### 💡 Insights - Pergunta 10 (Efetividade)

*(Preencher após execução)*

## 🎯 PERGUNTA 11: Insights Criativos

**Há algum padrão ou insight adicional não coberto pelas perguntas anteriores que mereça destaque?**

In [ ]:
print("💡 INSIGHTS CRIATIVOS E DESCOBERTAS ADICIONAIS")
print("="*80)

# 1. Análise por Gênero
if 'Gênero' in df.columns:
    print("\n👥 INSIGHT 1: Diferenças por Gênero")
    print("-"*80)
    
    genero_stats = df.groupby('Gênero')[indicadores_disp].mean()
    print(genero_stats)
    
    # Teste estatístico
    from scipy.stats import ttest_ind
    
    masc = df[df['Gênero'] == 'Masculino']['IDA'].dropna()
    fem = df[df['Gênero'] == 'Feminino']['IDA'].dropna()
    
    if len(masc) > 0 and len(fem) > 0:
        t_stat, p_value = ttest_ind(masc, fem)
        print(f"\nTeste t (IDA): t={t_stat:.4f}, p={p_value:.4f}")
        if p_value < 0.05:
            print("✅ Diferença estatisticamente significativa!")
        else:
            print("❌ Sem diferença estatisticamente significativa")

# 2. Análise Temporal (Ano de Ingresso)
if 'Ano ingresso' in df.columns:
    print("\n📅 INSIGHT 2: Evolução Temporal (Ano de Ingresso)")
    print("-"*80)
    
    temporal = df.groupby('Ano ingresso')[['IDA', 'IEG', 'IPV']].mean()
    print(temporal)

# 3. Perfil de Alunos Destaque
if 'Indicado' in df.columns:
    print("\n⭐ INSIGHT 3: Perfil dos Alunos Indicados/Destaque")
    print("-"*80)
    
    indicados = df[df['Indicado'] == 'Sim']
    nao_indicados = df[df['Indicado'] == 'Não']
    
    if len(indicados) > 0:
        print(f"\nAlunos Indicados: {len(indicados)} ({len(indicados)/len(df)*100:.1f}%)")
        print("\nMédia dos Indicadores:")
        print(indicados[indicadores_disp].mean())
        
        print("\nvs Não Indicados:")
        print(nao_indicados[indicadores_disp].mean())

# 4. Análise de Clusters (Perfis de Alunos)
print("\n🎯 INSIGHT 4: Perfis de Alunos (Clustering)")
print("-"*80)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Preparar dados
features_cluster = ['IDA', 'IEG', 'IPS']
X_cluster = df[features_cluster].dropna()

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# K-Means com 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Adicionar ao dataframe
X_cluster['Cluster'] = clusters

# Perfil de cada cluster
print("\nPerfil dos Clusters:")
cluster_profile = X_cluster.groupby('Cluster')[features_cluster].mean()
print(cluster_profile)

print("\nDistribuição dos Clusters:")
print(X_cluster['Cluster'].value_counts())

In [ ]:
# Visualização dos clusters
fig = px.scatter_3d(X_cluster, x='IDA', y='IEG', z='IPS',
                    color='Cluster',
                    title='Perfis de Alunos (3 Clusters)',
                    labels={'Cluster': 'Perfil'})
fig.show()

### 💡 Insights - Pergunta 11 (Insights Criativos)

**Principais Descobertas:**

1. **Diferenças por Gênero**: [Preencher após execução]
2. **Evolução Temporal**: [Preencher após execução]
3. **Perfil de Destaque**: [Preencher após execução]
4. **Clusters de Alunos**: Identificados 3 perfis distintos de alunos baseados em IDA, IEG e IPS

## 📝 Resumo Final das 11 Perguntas

### Respostas Consolidadas

1. **IAN (Adequação de Nível)**: [Resumo]
2. **IDA (Desempenho)**: [Resumo]
3. **IEG (Engajamento)**: Forte correlação com IDA (0.56) e IPV (0.59)
4. **IAA (Autoavaliação)**: [Resumo]
5. **IPS (Psicossocial)**: [Resumo]
6. **IPP (Psicopedagógico)**: [Resumo]
7. **IPV (Ponto de Virada)**: Fortemente influenciado por IDA e IEG
8. **INDE (Multidimensional)**: IDA e IEG explicam ~82% e ~80% da variância
9. **Modelo Preditivo**: ✅ Desenvolvido com sucesso (XGBoost, 3 classes)
10. **Efetividade**: [Resumo]
11. **Insights Criativos**: Identificados 3 perfis distintos de alunos

### Recomendações Principais

1. **Foco no Engajamento**: IEG é preditor forte de sucesso
2. **Intervenção Precoce**: Usar modelo preditivo para identificar riscos
3. **Acompanhamento Personalizado**: Adaptar estratégias por perfil de aluno
4. **Monitoramento Contínuo**: Avaliar progresso ao longo das fases

In [ ]:
print("✅ ANÁLISE COMPLETA FINALIZADA!")
print("\nPróximos passos:")
print("  1. Revisar insights e preencher resumos")
print("  2. Preparar apresentação")
print("  3. Gravar vídeo de demonstração")
print("  4. Deploy da aplicação")